# LoRA Fine-Tuning with DistilBERT on Emotion Dataset


## Objectives

- Load Emotion Dataset
- Tokenize Text
- Apply LoRA
- Fine-Tune DistilBERT
- Evaluate Performance
- Perform Inference


In [2]:
# !pip install transformers datasets peft accelerate evaluate scikit-learn -q

In [3]:
# PEFT now requires a newer TorchAO version.
# Colab may come with an older incompatible version.
!pip install -q --upgrade torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.6 MB/s eta 0:00:00:00:01


## Import Libraries

In [4]:

import torch
import numpy as np

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from sklearn.metrics import accuracy_score, f1_score


## Load Emotion Dataset

In [5]:
dataset = load_dataset("dair-ai/emotion")
dataset

README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

## View Sample

In [6]:
# Check the first 5 records
dataset["train"][0:5]

{'text': ['i didnt feel humiliated',
  'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake',
  'im grabbing a minute to post i feel greedy wrong',
  'i am ever feeling nostalgic about the fireplace i will know that it is still on the property',
  'i am feeling grouchy'],
 'label': [0, 0, 3, 2, 3]}

## Label Names

In [7]:
# check the names associated with each label
label_names = dataset["train"].features["label"].names

# Creates a list of integers: [0, 1, 2, ...]
encoded_values = [dataset["train"].features["label"].str2int(i) for i in dataset["train"].features["label"].names]

print(encoded_values)
print(label_names)

[0, 1, 2, 3, 4, 5]
['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


## Load Tokenizer

In [8]:
# tokenizer - autotokenizer will pick up automatically for distilBert
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Tokenization

In [9]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## Prepare Dataset

In [10]:
# preparing dataset
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

## Load Pretrained DistilBERT

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Configure LoRA

In [12]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

## Apply LoRA

In [13]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 742,662 || all params: 67,700,748 || trainable%: 1.0970


## Data Collator

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Evaluation Metrics

In [15]:

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "f1": f1
    }


## Training Arguments

In [16]:

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)


## Create Trainer

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


## Train Model

In [18]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.974498,0.829139,0.860000,0.860287
2,0.723717,0.641271,0.896000,0.895369


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=1000, training_loss=1.1449472274780272, metrics={'train_runtime': 190.8382, 'train_samples_per_second': 167.681, 'train_steps_per_second': 5.24, 'total_flos': 1078066446336000.0, 'train_loss': 1.1449472274780272, 'epoch': 2.0})

## Evaluate Model

In [19]:
results = trainer.evaluate()
results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6412709951400757,
 'eval_accuracy': 0.896,
 'eval_f1': 0.8953687429484941,
 'eval_runtime': 5.2342,
 'eval_samples_per_second': 382.106,
 'eval_steps_per_second': 12.036,
 'epoch': 2.0}

## Inference

In [20]:
id2label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

def predict_emotion(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    print("Text:", text)
    print("Predicted Emotion:", id2label[prediction])

## Test Predictions

In [22]:
predict_emotion("I feel very happy today")
predict_emotion("I am scared about the exam")
predict_emotion("This is extremely frustrating")

Text: I feel very happy today
Predicted Emotion: joy
Text: I am scared about the exam
Predicted Emotion: fear
Text: This is extremely frustrating
Predicted Emotion: sadness



# Key Takeaways

- LoRA trains only a small subset of parameters
- Pretrained weights remain frozen
- PEFT reduces GPU memory significantly
- Fine-tuning becomes faster and cheaper


**No tuning - only using distilBert**

In [23]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
training_args = TrainingArguments(
    output_dir="./results1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [26]:

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.511420,0.420841,0.917000,0.917626
2,0.241878,0.327165,0.931500,0.931248


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.5834653167724609, metrics={'train_runtime': 248.1807, 'train_samples_per_second': 128.938, 'train_steps_per_second': 4.029, 'total_flos': 1059814785024000.0, 'train_loss': 0.5834653167724609, 'epoch': 2.0})

In [27]:
results = trainer.evaluate()

results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.3271651864051819,
 'eval_accuracy': 0.9315,
 'eval_f1': 0.9312484466971557,
 'eval_runtime': 5.0471,
 'eval_samples_per_second': 396.268,
 'eval_steps_per_second': 12.482,
 'epoch': 2.0}

In [28]:
predict_emotion("I feel very happy today")

predict_emotion("I am scared about the exam")

predict_emotion("This is extremely frustrating")

Text: I feel very happy today
Predicted Emotion: joy
Text: I am scared about the exam
Predicted Emotion: fear
Text: This is extremely frustrating
Predicted Emotion: anger
